# RL Algorithms Comparison: DQN vs REINFORCE on CartPole-v1

This notebook implements and compares two Reinforcement Learning algorithms, **DQN (Deep Q-Network)** and **REINFORCE**, on the `CartPole-v1` environment. 

The implementation follows specific requirements:
- **DQN**: Uses a Replay Buffer and Target Network.
- **REINFORCE**: Uses Policy Gradient with a baseline.
- **Comparison**: Both are evaluated over multiple runs to ensure statistical significance.

## Hyperparameters (from requirements)
**DQN**:
- Hidden Layer Size: 24
- Learning Rate: 0.001
- Replay Buffer Size: 10000
- Target Update Frequency: 50 steps
- Epsilon Decay: 1.0 to 0.01 over 500 episodes

**REINFORCE**:
- Hidden Layer Size: 24
- Learning Rate: 0.005
- Baseline: Mean reward (simple baseline for stability)

In [ ]:
import gymnasium as gym  # gymnasium — библиотека для сред OpenAI Gym (среды RL)
import torch  # PyTorch — основной фреймворк для тензоров и нейросетей
import torch.nn as nn  # Подмодуль для определения архитектуры нейронных сетей
import torch.optim as optim  # Оптимизаторы (SGD, Adam и т.д.) для обучения моделей
import numpy as np  # numpy — библиотека для численных операций и массивов
import random  # Модуль для работы со случайностью (используется для reproducibility)
from collections import deque  # deque — быстрая очередь с ограничением по длине, используется для ReplayBuffer
import matplotlib.pyplot as plt  # matplotlib — визуализация графиков и результатов

# Устанавливаем случайные семена для воспроизводимости результатов (important for experiments)
def set_seed(seed):  # Функция задаёт одно и то же начальное состояние генераторов случайных чисел
    random.seed(seed)  # Задаём seed для встроенного модуля random (python)
    np.random.seed(seed)  # Задаём seed для numpy
    torch.manual_seed(seed)  # Задаём seed для PyTorch (CPU)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Выбираем устройство: GPU если доступно, иначе CPU
print(f"Using device: {device}")  # Печатаем выбранное устройство для отладки

Using device: cuda


## 1. Deep Q-Network (DQN) Implementation

The DQN agent uses a Q-Network to approximate the Q-value function. It utilizes a Replay Buffer to store transitions and a Target Network to stabilize training.

In [ ]:
class QNetwork(nn.Module):  # Класс нейронной сети для приближения Q-функции
    def __init__(self, state_dim, action_dim, hidden_size=24):  # Инициализация сети с размерами входа/выхода и скрытого слоя
        super(QNetwork, self).__init__()  # Вызов конструктора родителя nn.Module
        self.fc1 = nn.Linear(state_dim, hidden_size)  # Первый полносвязный слой: состояние -> скрытый слой
        self.fc2 = nn.Linear(hidden_size, action_dim)  # Выходной слой: скрытый слой -> значения Q для каждой акции
        self.relu = nn.ReLU()  # Функция активации ReLU для нелинейности

    def forward(self, x):  # Прямой проход сети — вычисляет Q-значения для входного состояния
        x = self.relu(self.fc1(x))  # Пропуск через первый слой и активацию
        return self.fc2(x)  # Возвращаем Q-логиты (без активации — регрессия)

class ReplayBuffer:  # Буфер воспроизведения — хранит прошлые переходы для мини-батчей
    def __init__(self, capacity):  # Инициализация с максимальной ёмкостью
        self.buffer = deque(maxlen=capacity)  # Используем deque для быстрого добавления и автоматического удаления старых элементов

    def push(self, state, action, reward, next_state, done):  # Добавляет переход в буфер
        self.buffer.append((state, action, reward, next_state, done))  # Храним весь переход как кортеж

    def sample(self, batch_size):  # Случайная выборка батча переходов
        batch = random.sample(self.buffer, batch_size)  # Берём случайные элементы для обучения (разнообразие опыта)
        state, action, reward, next_state, done = zip(*batch)  # Распаковываем кортежи в отдельные списки
        return (
            torch.FloatTensor(np.array(state)).to(device),  # Преобразуем состояния в FloatTensor и отправляем на device
            torch.LongTensor(action).to(device),  # Действия храним как LongTensor (для индексации)
            torch.FloatTensor(reward).to(device),  # Награды как FloatTensor
            torch.FloatTensor(np.array(next_state)).to(device),  # Следующие состояния как FloatTensor
            torch.FloatTensor(done).to(device)  # Флаги завершения эпизода как FloatTensor (0/1)
        )

    def __len__(self):  # Возвращает текущий размер буфера
        return len(self.buffer)  # Длина очереди (количество записанных переходов)

def train_dqn(seed, episodes=500):  # Основная функция обучения DQN с заданным seed и числом эпизодов
    set_seed(seed)  # Устанавливаем глобальные семена для воспроизводимости
    env = gym.make('CartPole-v1')  # Создаём среду CartPole-v1
    
    state_dim = env.observation_space.shape[0]  # Размер векторного представления состояния среды
    action_dim = env.action_space.n  # Количество дискретных действий в среде
    
    # Hyperparameters  # Параметры обучения и структуры сети
    hidden_size = 24  # Размер скрытого слоя сети
    lr = 0.001  # Скорость обучения (learning rate) для оптимизатора
    buffer_size = 10000  # Максимальный размер replay buffer'а
    batch_size = 32  # Размер мини-батча для обновления весов
    gamma = 0.99  # Коэффициент дисконтирования вознаграждений
    target_update_freq = 50  # Частота обновления target-сети по шагам
    epsilon_start = 1.0  # Начальное значение эпсилон для epsilon-greedy
    epsilon_end = 0.01  # Конечное значение эпсилон после decay
    epsilon_decay = (epsilon_start - epsilon_end) / episodes  # Размер уменьшения эпсилон за эпизод

    policy_net = QNetwork(state_dim, action_dim, hidden_size).to(device)  # Основная Q-сеть (policy_net)
    target_net = QNetwork(state_dim, action_dim, hidden_size).to(device)  # Target сеть для стабильности обучения
    target_net.load_state_dict(policy_net.state_dict())  # Инициализируем target сеть весами policy сети
    target_net.eval()  # Переводим target сеть в режим оценки (без обучения)

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)  # Оптимизатор Adam для настройки весов policy_net
    memory = ReplayBuffer(buffer_size)  # Инициализация replay buffer'а

    epsilon = epsilon_start  # Текущее эпсилон значение для эпсилон-greedy стратегии
    rewards_history = []  # Список для хранения суммарных вознаграждений по эпизодам
    steps_done = 0  # Счётчик шагов для обновления target сети

    for episode in range(episodes):  # Цикл по эпизодам обучения
        state, _ = env.reset(seed=seed + episode) # Вариация seed по эпизоду — иногда используют для дополнительного рандома
        # Комментарии о seed: оставляем глобальную инициализацию, но здесь можно делать варьирование seed'а
        state, _ = env.reset()  # Дополнительный reset для текущего эпизода (форма возвращаемого значения)
        
        total_reward = 0  # Накопленная награда за текущий эпизод
        done = False  # Флаг завершения эпизода
        
        while not done:  # Шаги внутри эпизода до окончания
            # Epsilon-greedy action selection  # Выбор действия: исследование или использование сети
            if random.random() < epsilon:  # С вероятностью epsilon берём случайное действие
                action = env.action_space.sample()  # Случайное действие из пространства действий
            else:
                with torch.no_grad():  # Без вычисления градиентов — только вывод сети
                    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)  # Переводим состояние в тензор и добавляем batch-dim
                    action = policy_net(state_tensor).argmax().item()  # Выбираем действие с максимальным Q-value

            next_state, reward, terminated, truncated, _ = env.step(action)  # Применяем действие в среде и получаем отклик
            done = terminated or truncated  # Эпизод завершён если терминал или усечение
            
            memory.push(state, action, reward, next_state, done)  # Сохраняем переход в буфер
            state = next_state  # Переход к следующему состоянию
            total_reward += reward  # Накопление награды за эпизод
            steps_done += 1  # Увеличиваем счётчик шагов

            # Training step  # Обучающий шаг: обновляем policy_net, если достаточный опыт в буфере
            if len(memory) >= batch_size:  # Проверяем, есть ли достаточно переходов для батча
                states, actions, rewards, next_states, dones = memory.sample(batch_size)  # Получаем случайный батч

                q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)  # Вычисляем Q(s,a) для выбранных действий
                with torch.no_grad():  # Вычисления для цели без градиентов (target)
                    next_q_values = target_net(next_states).max(1)[0]  # Берём max Q для следующего состояния (целевые значения)
                    target_q_values = rewards + gamma * next_q_values * (1 - dones)  # Формируем TD-таргеты

                loss = nn.MSELoss()(q_values, target_q_values)  # MSE loss между предсказанными и целевыми Q-значениями

                optimizer.zero_grad()  # Обнуляем градиенты оптимизатора
                loss.backward()  # Прямое распространение ошибки, вычисляем градиенты
                optimizer.step()  # Шаг оптимизации — обновляем веса сети

            # Update target network  # При необходимости — синхронизируем target-сеть с policy-сетью
            if steps_done % target_update_freq == 0:
                target_net.load_state_dict(policy_net.state_dict())  # Копируем веса

        # Decay epsilon  # Плавное уменьшение epsilon для уменьшения исследования во времени
        epsilon = max(epsilon_end, epsilon - epsilon_decay)  # Ограничиваем снизу epsilon_end
        rewards_history.append(total_reward)  # Сохраняем результат эпизода
        
        if (episode + 1) % 50 == 0:  # Периодическая печать информации о прогрессе
            print(f"DQN Episode {episode+1}/{episodes}, Reward: {total_reward}, Epsilon: {epsilon:.2f}")

    return rewards_history  # Возвращаем список вознаграждений для анализа

## 2. REINFORCE Implementation

The REINFORCE algorithm optimizes the policy directly. It collects a full trajectory and then updates the policy to increase the probability of actions that led to high returns.

In [ ]:
class PolicyNetwork(nn.Module):  # Нейросеть политики: принимает состояние и возвращает вероятности действий
    def __init__(self, state_dim, action_dim, hidden_size=24):  # Инициализация архитектуры сети
        super(PolicyNetwork, self).__init__()  # Вызов конструктора базового класса
        self.fc1 = nn.Linear(state_dim, hidden_size)  # Первый полносвязный слой
        self.fc2 = nn.Linear(hidden_size, action_dim)  # Выходной слой превращает скрытое представление в логиты по акциям
        self.relu = nn.ReLU()  # ReLU активация для скрытых слоёв
        self.softmax = nn.Softmax(dim=1)  # Softmax для получения вероятностного распределения действий по батчу

    def forward(self, x):  # Прямой проход: возвращает распределение вероятностей действий
        x = self.relu(self.fc1(x))  # Пропускаем через первый слой и ReLU
        x = self.fc2(x)  # Получаем логиты для действий
        return self.softmax(x)  # Возвращаем вероятности действий после Softmax

def train_reinforce(seed, episodes=500):  # Функция обучения REINFORCE (политик-градиент)
    set_seed(seed)  # Устанавливаем семя для воспроизводимости
    env = gym.make('CartPole-v1')  # Создаём среду CartPole
    
    state_dim = env.observation_space.shape[0]  # Размерность входного состояния
    action_dim = env.action_space.n  # Количество дискретных действий
    
    # Hyperparameters  # Параметры обучения политики
    hidden_size = 24  # Размер скрытого слоя
    lr = 0.005  # Скорость обучения для оптимизатора
    gamma = 0.99  # Дисконтирование будущих наград

    policy_net = PolicyNetwork(state_dim, action_dim, hidden_size).to(device)  # Инициализация сети политики и перенос на device
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)  # Оптимизатор Adam для обновления параметров политики

    rewards_history = []  # Массив для хранения суммарных наград по эпизодам

    for episode in range(episodes):  # Цикл по эпизодам
        state, _ = env.reset()  # Сброс среды и получение начального состояния
        log_probs = []  # Список логарифмов вероятностей выбранных действий (для вычисления градиента)
        rewards = []  # Список наград для эпизода
        done = False  # Флаг окончания эпизода

        while not done:  # Пока эпизод не завершается
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)  # Преобразуем состояние в тензор и добавляем batch-ось
            probs = policy_net(state_tensor)  # Получаем вероятности действий от сети
            
            # Sample action from distribution  # Выбор действия по вероятностному распределению
            dist = torch.distributions.Categorical(probs)  # Создаём категориальное распределение на основе вероятностей
            action = dist.sample()  # Семплируем действие из распределения
            log_prob = dist.log_prob(action)  # Сохраняем логарифм вероятности выбранного действия
            
            next_state, reward, terminated, truncated, _ = env.step(action.item())  # Выполняем действие в среде и получаем результат
            done = terminated or truncated  # Эпизод завершён по сигналу среды
            
            log_probs.append(log_prob)  # Сохраняем логарифм вероятности для обновления политики
            rewards.append(reward)  # Добавляем полученную награду в список эпизодических наград
            state = next_state  # Переходим к следующему состоянию

        # Calculate returns  # Вычисление накопленных доходов G_t в обратном порядке
        returns = []  # Список для хранений возвращённых сумм вознаграждений
        G = 0  # Инициализация накопленного вознаграждения
        for r in reversed(rewards):  # Идём по вознаграждениям в обратном порядке
            G = r + gamma * G  # Обновляем G с учётом дисконтирования
            returns.insert(0, G)  # Вставляем значение в начало списка returns
        
        returns = torch.FloatTensor(returns).to(device)  # Переводим returns в Tensor и на device
        # Baseline: Normalize returns to reduce variance  # Нормализация для снижения дисперсии градиента
        # returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        # Text suggests simple baseline or just raw. Let's stick to standard REINFORCE or simple normalization if unstable.
        # The text mentions "baseline (baseline) for reducing variance". 
        # A common simple baseline is subtracting the mean.
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)  # Нормализуем returns — вычитаем среднее и делим на std

        policy_loss = []  # Список элементов потерь для последующего суммирования
        for log_prob, G in zip(log_probs, returns):  # Для каждого шага в эпизоде
            policy_loss.append(-log_prob * G)  # Элемент потери: -logπ(a|s) * G (умноженная награда)

        optimizer.zero_grad()  # Обнуляем градиенты перед обратным проходом
        policy_loss = torch.cat(policy_loss).sum()  # Склеиваем и суммируем элементы потерь
        policy_loss.backward()  # Вычисляем градиенты для параметров политики
        optimizer.step()  # Обновляем параметры сети политики

        total_reward = sum(rewards)  # Суммарная награда за эпизод
        rewards_history.append(total_reward)  # Сохранение результата

        if (episode + 1) % 50 == 0:  # Тайпичная логика — печатать прогресс раз в N эпизодов
            print(f"REINFORCE Episode {episode+1}/{episodes}, Reward: {total_reward}")

    return rewards_history  # Возвращаем историю вознаграждений для анализа и сравнения

## 3. Experiments and Comparison

We will run both algorithms for 10 independent runs (seeds 0-9) to gather statistical data.

In [ ]:
num_runs = 1  # Количество независимых прогонов (разных seed'ов) для статистики
episodes = 100  # Количество эпизодов в каждом прогона

dqn_results = []  # Список для сохранения результатов DQN (по каждому прогону)
reinforce_results = []  # Список для сохранения результатов REINFORCE

print("Starting DQN Experiments...")  # Выводим статус запуска экспериментов DQN
for seed in range(num_runs):  # Цикл по разным seed'ам для статистики
    print(f"--- DQN Run {seed+1}/{num_runs} ---")  # Печатаем текущий прогон
    dqn_results.append(train_dqn(seed, episodes))  # Запускаем обучение DQN и добавляем историю вознаграждений в список

print("\nStarting REINFORCE Experiments...")  # Переход к запуску экспериментов для REINFORCE
for seed in range(num_runs):  # Для каждого seed запускаем отдельный прогон для REINFORCE
    print(f"--- REINFORCE Run {seed+1}/{num_runs} ---")  # Печать прогресса
    reinforce_results.append(train_reinforce(seed, episodes))  # Запуск и сохранение результатов

Starting DQN Experiments...
--- DQN Run 1/1 ---
DQN Episode 50/100, Reward: 17.0, Epsilon: 0.50
DQN Episode 100/100, Reward: 31.0, Epsilon: 0.01

Starting REINFORCE Experiments...
--- REINFORCE Run 1/1 ---
REINFORCE Episode 50/100, Reward: 73.0
REINFORCE Episode 100/100, Reward: 157.0


In [ ]:
# Convert to numpy arrays for easier analysis  # Переводим списки результатов в numpy массивы для удобства обработки
dqn_results = np.array(dqn_results)  # Массив shape=(num_runs, episodes) для DQN
reinforce_results = np.array(reinforce_results)  # Массив shape=(num_runs, episodes) для REINFORCE

# Calculate means and standard deviations  # Рассчитываем среднее и СК по прогону по эпизодам
dqn_mean = np.mean(dqn_results, axis=0)  # Среднее по прогону для каждого эпизода (DQN)
dqn_std = np.std(dqn_results, axis=0)  # Стандартное отклонение по прогону для DQN

reinforce_mean = np.mean(reinforce_results, axis=0)  # Среднее для REINFORCE
reinforce_std = np.std(reinforce_results, axis=0)  # СК для REINFORCE

# Plotting  # Построение графиков сравнения методов
plt.figure(figsize=(12, 6))  # Создаём область для графика

# DQN Plot  # Рисуем среднюю график DQN
plt.plot(dqn_mean, label='DQN', color='blue')  # Линия среднего DQN
plt.fill_between(range(episodes), dqn_mean - dqn_std, dqn_mean + dqn_std, color='blue', alpha=0.2)  # Заливка ±1σ

# REINFORCE Plot  # Рисуем среднюю график REINFORCE
plt.plot(reinforce_mean, label='REINFORCE', color='orange')  # Линия среднего REINFORCE
plt.fill_between(range(episodes), reinforce_mean - reinforce_std, reinforce_mean + reinforce_std, color='orange', alpha=0.2)  # Заливка ±1σ

plt.title('DQN vs REINFORCE on CartPole-v1 (10 Runs)')  # Заголовок графика
plt.xlabel('Episode')  # Подпись оси X
plt.ylabel('Average Reward')  # Подпись оси Y
plt.legend()  # Показываем легенду
plt.grid(True)  # Включаем сетку для читабельности
plt.show()  # Отображаем график

# Final Statistics  # Выводим итоговые статистики за последние N эпизодов (например, последние 50)
print(f"DQN Final Average Reward (Last 50 eps): {np.mean(dqn_results[:, -50:]):.2f} +/- {np.std(dqn_results[:, -50:]):.2f}")  # Среднее и СК для DQN за последние 50 эпизодов
print(f"REINFORCE Final Average Reward (Last 50 eps): {np.mean(reinforce_results[:, -50:]):.2f} +/- {np.std(reinforce_results[:, -50:]):.2f}")  # То же для REINFORCE